# 04 — Citations and Abstention: Make Evidence and Uncertainty Explicit

**Track:** Beginner · **Stage:** Generation & Guardrails

An answer can contain a link and still be unsupported. A trustworthy system preserves evidence identity through retrieval, context construction, generation, validation, and rendering. 

In this notebook, you will build a prompt-level citation convention and implement application handling for a model-produced abstention signal.

> **Note on Mock Models:** For deterministic learning without API keys, this lab uses a `FakeListLLM`. The pre-programmed responses test the **application control flow**, not the model's intelligence. Supplying a fake response does not prove a real LLM would follow the instruction perfectly—it proves our application can handle both success and failure states safely.

## Setup: Dependencies

We use standard Python dataclasses/Pydantic for structured decisions and LangChain for basic abstractions.

In [1]:
# !pip install langchain-core pydantic

from typing import Literal, List, Optional
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_core.language_models import FakeListLLM


## 1. The Corpus and Request-Local Identity

Assume retrieval has already happened. We have two documents in our context window.

Instead of passing raw filenames to the model (which the model might hallucinate or corrupt), we will assign a simple, request-local evidence ID (`E1`, `E2`) to each retrieved document. The model only needs to output these short IDs, and the application will map them back to the rich source metadata later.

In [2]:
retrieved_docs = [
    Document(
        page_content="Enterprise customers receive a status update within 30 minutes of a confirmed P1 incident.",
        metadata={"source": "sla_policy.md", "title": "Incident Communication SLA", "chunk_id": "policy-sla#p1-update"}
    ),
    Document(
        page_content="To page the on-call engineer, use the /page command in the #incidents Slack channel.",
        metadata={"source": "incident_response.md", "title": "On-Call Procedures", "chunk_id": "incident-resp#pager"}
    )
]

# The application holds the mapping between the short ID and the full metadata
evidence_map = {
    "E1": retrieved_docs[0],
    "E2": retrieved_docs[1]
}

def format_docs_for_prompt(evidence_dict):
    formatted = []
    for e_id, doc in evidence_dict.items():
        formatted.append(f"<EVIDENCE id='{e_id}'>\n{doc.page_content}\n</EVIDENCE>")
    return "\n\n".join(formatted)

print(format_docs_for_prompt(evidence_map))


<EVIDENCE id='E1'>
Enterprise customers receive a status update within 30 minutes of a confirmed P1 incident.
</EVIDENCE>

<EVIDENCE id='E2'>
To page the on-call engineer, use the /page command in the #incidents Slack channel.
</EVIDENCE>


## 2. Baseline A: The String Sentinel

A common teaching pattern is to instruct the model to output a specific string like `INSUFFICIENT_EVIDENCE` if it cannot answer the question. This demonstrates that model output can change application behavior.

In [3]:
# Scenario: Model correctly outputs the sentinel
fake_response = "INSUFFICIENT_EVIDENCE"

if "INSUFFICIENT_EVIDENCE" in fake_response:
    print("Application Action: Abstain and escalate to human.")
else:
    print("Application Action: Render answer.")


Application Action: Abstain and escalate to human.


**Why is this brittle?**
String matching fails gracefully. If the model outputs `The answer is INSUFFICIENT_EVIDENCE according to the docs`, the string match still triggers, even though the model didn't cleanly abstain. We need a stronger architecture.

## 3. Structured Decisions

Instead of parsing raw strings, a production RAG application uses structured outputs (e.g., via OpenAI Tool Calling or JSON schema).

The learner should understand this architecture:
```text
LLM / decision producer
        ↓
structured RAGDecision
        ↓
deterministic validation
        ↓
render / abstain / block
```

In [4]:
class RAGDecision(BaseModel):
    decision: Literal["answer", "insufficient_evidence", "conflicting_evidence"] = Field(
        description="The type of response the model is providing."
    )
    answer: Optional[str] = Field(
        default=None, 
        description="The answer to the user's question, if applicable."
    )
    citations: List[str] = Field(
        default_factory=list, 
        description="A list of evidence IDs (e.g., 'E1') that support the answer."
    )
    reason: Optional[str] = Field(
        default=None, 
        description="Internal reasoning for abstention or conflicts."
    )


## 4. Deterministic Validation

Before we show an answer to a user, the application must validate the decision. 
For example, if the model decides to "answer", it MUST provide citations, and those citations MUST actually exist in the `evidence_map`.

In [5]:
def validate_decision(decision: RAGDecision, evidence_map: dict) -> list[str]:
    errors = []
    
    if decision.decision == "answer":
        if not decision.answer:
            errors.append("Decision is 'answer' but no answer text was provided.")
        if not decision.citations:
            errors.append("Decision is 'answer' but no citations were provided.")
            
        # Check for invented citations
        for citation in decision.citations:
            if citation not in evidence_map:
                errors.append(f"Invented Citation: '{citation}' does not exist in the evidence map.")
                
    elif decision.decision == "insufficient_evidence":
        if decision.answer:
            errors.append("Decision is 'insufficient_evidence' but an answer was provided.")
            
    # If errors exist, the application should block the response or fall back gracefully
    return errors


## 5. Separate Rendering Step

The model should not generate arbitrary source URLs or document metadata. The application already knows those values. Render them only after validation.

In [6]:
def render_response(decision: RAGDecision, evidence_map: dict):
    print("--- Final Rendered Output ---")
    if decision.decision == "insufficient_evidence":
        print("Assistant: I don't have enough information in my knowledge base to answer that safely.")
    elif decision.decision == "conflicting_evidence":
        print("Assistant: The retrieved documents contain conflicting information. Please verify with a human.")
    elif decision.decision == "answer":
        print(f"Assistant: {decision.answer}\n")
        print("Sources:")
        # Render the human-readable provenance
        for citation in set(decision.citations): # deduplicate citations
            doc = evidence_map[citation]
            print(f" - [{citation}] {doc.metadata['title']} ({doc.metadata['source']})")
    print("-----------------------------\n")

# A helper to run our pipeline
def run_pipeline(decision: RAGDecision, evidence_map: dict):
    print(f"1. Model Output: {decision.model_dump_json(indent=2)}")
    errors = validate_decision(decision, evidence_map)
    if errors:
        print(f"2. Validation Failed! Errors:")
        for e in errors:
            print(f"   - {e}\n")
    else:
        print("2. Validation Passed!\n")
        render_response(decision, evidence_map)


## 6. Lab Scenarios

Let's test our application control flow against different model outputs.

### Scenario A — Supported Answer
The evidence clearly answers the question, and the model correctly cites `E1`.

In [7]:
decision_A = RAGDecision(
    decision="answer",
    answer="Enterprise customers receive updates within 30 minutes.",
    citations=["E1"]
)
run_pipeline(decision_A, evidence_map)


1. Model Output: {
  "decision": "answer",
  "answer": "Enterprise customers receive updates within 30 minutes.",
  "citations": [
    "E1"
  ],
  "reason": null
}
2. Validation Passed!

--- Final Rendered Output ---
Assistant: Enterprise customers receive updates within 30 minutes.

Sources:
 - [E1] Incident Communication SLA (sla_policy.md)
-----------------------------



### Scenario B — Insufficient Evidence
Question: *What is the update SLA for standard-tier customers?*
No evidence answers this. The model cleanly abstains.

In [8]:
decision_B = RAGDecision(
    decision="insufficient_evidence",
    reason="The context only mentions Enterprise customers, not standard-tier."
)
run_pipeline(decision_B, evidence_map)


1. Model Output: {
  "decision": "insufficient_evidence",
  "answer": null,
  "citations": [],
  "reason": "The context only mentions Enterprise customers, not standard-tier."
}
2. Validation Passed!

--- Final Rendered Output ---
Assistant: I don't have enough information in my knowledge base to answer that safely.
-----------------------------



### Scenario C — Invented Citation (Hallucination)
The model attempts to cite `E99`, which was never provided in the context. Our deterministic validation catches this.

In [9]:
decision_C = RAGDecision(
    decision="answer",
    answer="Enterprise customers receive updates within 30 minutes.",
    citations=["E99"]
)
run_pipeline(decision_C, evidence_map)


1. Model Output: {
  "decision": "answer",
  "answer": "Enterprise customers receive updates within 30 minutes.",
  "citations": [
    "E99"
  ],
  "reason": null
}
2. Validation Failed! Errors:
   - Invented Citation: 'E99' does not exist in the evidence map.



### Scenario D — Valid but Incorrect Citation
The model cites `E2`, which *is* a valid evidence ID, but `E2` is about paging the on-call engineer, not the 30-minute SLA!

**Key concept:** `E2` exists (validity = yes), but `E2` does not support the claim (correctness = no). 
Deterministic validation cannot easily catch this semantic error. This requires downstream evaluation (e.g., an LLM-as-a-judge checking for entailment).

In [10]:
decision_D = RAGDecision(
    decision="answer",
    answer="Enterprise customers receive updates within 30 minutes.",
    citations=["E2"]
)
run_pipeline(decision_D, evidence_map)


1. Model Output: {
  "decision": "answer",
  "answer": "Enterprise customers receive updates within 30 minutes.",
  "citations": [
    "E2"
  ],
  "reason": null
}
2. Validation Passed!

--- Final Rendered Output ---
Assistant: Enterprise customers receive updates within 30 minutes.

Sources:
 - [E2] On-Call Procedures (incident_response.md)
-----------------------------



### Scenario E — Citation Completeness
The model makes two claims, but only cites one.
1. Enterprise customers receive updates in 30 minutes. (Supported by E1)
2. The incident commander must personally send the update. (Unsupported hallucination)

In [11]:
decision_E = RAGDecision(
    decision="answer",
    answer="Enterprise customers receive updates within 30 minutes. The incident commander must personally send the update.",
    citations=["E1"]
)
run_pipeline(decision_E, evidence_map)


1. Model Output: {
  "decision": "answer",
  "answer": "Enterprise customers receive updates within 30 minutes. The incident commander must personally send the update.",
  "citations": [
    "E1"
  ],
  "reason": null
}
2. Validation Passed!

--- Final Rendered Output ---
Assistant: Enterprise customers receive updates within 30 minutes. The incident commander must personally send the update.

Sources:
 - [E1] Incident Communication SLA (sla_policy.md)
-----------------------------



Citation presence = yes. Validity = yes. Completeness = no! Claim-level decomposition is an advanced technique to catch this.

### Scenario F — Conflicting Evidence

Often, retrieval will return two documents that contradict each other. Here, rich metadata is crucial. If the application cannot resolve the conflict using metadata (like `version` or `effective_date`), the safe action is to trigger the `conflicting_evidence` decision type rather than letting the model silently choose one.

In [12]:
conflict_docs = [
    Document(
        page_content="P1 customers receive updates within 60 minutes.",
        metadata={"source": "sla_policy.md", "title": "Incident Communication SLA", "version": "1"}
    ),
    Document(
        page_content="P1 customers receive updates within 30 minutes.",
        metadata={"source": "sla_policy.md", "title": "Incident Communication SLA", "version": "2"}
    )
]

conflict_map = {
    "E1": conflict_docs[0],
    "E2": conflict_docs[1]
}

decision_F = RAGDecision(
    decision="conflicting_evidence",
    reason="E1 says 60 minutes, but E2 says 30 minutes."
)
run_pipeline(decision_F, conflict_map)


1. Model Output: {
  "decision": "conflicting_evidence",
  "answer": null,
  "citations": [],
  "reason": "E1 says 60 minutes, but E2 says 30 minutes."
}
2. Validation Passed!

--- Final Rendered Output ---
Assistant: The retrieved documents contain conflicting information. Please verify with a human.
-----------------------------



## Reflection

1. **Groundedness vs. Factual Correctness vs. Freshness**: If the model cites `E1` (Version 1, 60 minutes), the answer is *grounded* in the retrieved evidence, but it lacks *freshness* and is factually *incorrect* according to the current policy. Citation does not guarantee truth.
2. **Authorization**: Citation validation cannot repair an authorization failure that already exposed restricted evidence to the model. Access controls must happen *before* retrieval.
